# P-T Profile Explorer

## Line et al. 2013

Three-channel radiative equilibrium profile.

$$T^4(\tau) = \frac{3T_{\rm int}^4}{4}\left(\frac{2}{3} + \tau\right) + \frac{3T_{\rm irr}^4}{4}\left[(1-\alpha)\,\xi_1 + \alpha\,\xi_2\right]$$

$$\tau(P) = \frac{\kappa_{\rm IR}\,P_{\rm ref}}{g\,n}\,P^n \qquad \xi_i(\gamma_i,\tau) = \frac{2}{3} + \frac{2}{3\gamma_i}\!\left[1 + \left(\frac{\gamma_i\tau}{2}-1\right)e^{-\gamma_i\tau}\right] + \frac{2\gamma_i}{3}\!\left(1-\frac{\tau^2}{2}\right)E_2(\gamma_i\tau)$$

where $\gamma_i = \kappa_{{\rm vis},i}/\kappa_{\rm IR}$ and $E_2$ is the second exponential integral.

---

## petitRADTRANS (Mollière+2019, Eqs. 15–16)

Guillot (2010) base profile, modified at high altitudes and boxcar-smoothed:

$$T_{\rm Guillot}^4(P) = \frac{3T_{\rm int}^4}{4}\!\left(\frac{2}{3}+\delta P\right) + \frac{3T_{\rm eq}^4}{4}\!\left[\frac{2}{3}+\frac{1}{\gamma\sqrt{3}}+\left(\frac{\gamma}{\sqrt{3}}-\frac{1}{\gamma\sqrt{3}}\right)e^{-\gamma\delta\sqrt{3}\,P}\right]$$

$$T(P) = \left\langle T_{\rm Guillot}(P)\cdot\left(1 - \frac{\alpha}{1 + P/P_{\rm trans}}\right)\right\rangle_{\!\Delta\log P\,=\,1.25\,\rm dex}$$

where $\delta = \kappa_{\rm IR}/g$, $\gamma$ is the visible-to-IR opacity ratio, $\alpha$ controls upper-atmosphere non-isothermality, and $P_{\rm trans}$ sets the transition pressure.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import expn

plt.style.use('science.mplstyle')

# ── Line+2013 P-T profile ─────────────────────────────────────────────────────
# T^4(τ) = 3T_int^4/4 · (2/3 + τ)  +  3T_irr^4/4 · [(1-α)ξ₁ + α·ξ₂]
# τ(P)   = (κ · P_ref) / (g · n) · P^n
# ξᵢ    = two-stream penetration function for opacity ratio γᵢ

_BAR_TO_PA = 1e5

def compute_optical_depth(pressure_bar, kappa_ir_m2_kg, gravity_m_s2, power_law_n):
    tau_scale = (kappa_ir_m2_kg * _BAR_TO_PA) / gravity_m_s2
    return (tau_scale / power_law_n) * pressure_bar ** power_law_n

def compute_xi(gamma, tau):
    gt = gamma * tau
    return (2/3
            + (2/(3*gamma)) * (1 + (gt/2 - 1) * np.exp(-gt))
            + (2*gamma/3)   * (1 - 0.5*tau**2) * expn(2, gt))

def apply_convective_adjustment(pressure_bar, temperature_k, adiabatic_gradient):
    adjusted = np.copy(temperature_k)
    log_p    = np.log10(pressure_bar)
    log_t    = np.log10(temperature_k)
    for i in range(1, len(adjusted)):
        if (log_t[i] - log_t[i-1]) / (log_p[i] - log_p[i-1]) > adiabatic_gradient:
            adjusted[i:] = adjusted[i-1] * (pressure_bar[i:] / pressure_bar[i-1]) ** adiabatic_gradient
            break
    return adjusted

def line2013_profile(pressure_bar, t_int_k, t_irr_k,
                     kappa_ir_m2_kg, power_law_n,
                     gamma_1, gamma_2, alpha,
                     gravity_m_s2=25.0,
                     adiabatic_gradient=None,
                     temperature_shift_k=0.0):
    tau = compute_optical_depth(pressure_bar, kappa_ir_m2_kg, gravity_m_s2, power_law_n)
    xi1 = compute_xi(gamma_1, tau)
    xi2 = compute_xi(gamma_2, tau)
    t4  = (3*t_int_k**4/4) * (2/3 + tau) + (3*t_irr_k**4/4) * ((1-alpha)*xi1 + alpha*xi2)
    T   = np.clip(t4, 0.0, None) ** 0.25
    if adiabatic_gradient is not None:
        T = apply_convective_adjustment(pressure_bar, T, adiabatic_gradient)
    return T + temperature_shift_k

print("Line+2013 functions loaded.")


In [ ]:
from scipy.ndimage import uniform_filter1d

# ── petitRADTRANS PT profile (Mollière+2019, Eqs. 15–16) ─────────────────────
# T_Guillot^4 = 3T_int^4/4 * (2/3 + δP)
#             + 3T_eq^4/4  * [2/3 + 1/(γ√3) + (γ/√3 − 1/(γ√3)) e^{−γδ√3 P}]
#
# T(P) = ⟨ T_Guillot(P) · (1 − α / (1 + P/P_trans)) ⟩_{Δlog P = 1.25 dex}
#
# δ = κ_IR / g   (τ = δ · P,  P in bar with δ absorbing unit conversion)

def _guillot(pressure_bar, delta, gamma, T_int, T_eq):
    tau = delta * pressure_bar
    T4  = (
        3/4 * T_int**4 * (2/3 + tau)
        + 3/4 * T_eq**4 * (
            2/3
            + 1 / (gamma * np.sqrt(3))
            + (gamma / np.sqrt(3) - 1 / (gamma * np.sqrt(3))) * np.exp(-gamma * np.sqrt(3) * tau)
        )
    )
    return np.clip(T4, 0.0, None) ** 0.25


def _boxcar_logP(pressure_bar, T, width_dex=1.25):
    """Running mean over a fixed window in log10(P) space."""
    dp = abs(np.log10(pressure_bar[1]) - np.log10(pressure_bar[0]))  # uniform log-spacing
    n  = max(1, int(round(width_dex / dp)))
    return uniform_filter1d(T, size=n, mode='nearest')


def molliere2019_pt_profile(pressure_bar, log_delta, log_gamma, T_int, T_eq, alpha, log_P_trans):
    """
    Mollière+2019 (petitRADTRANS) PT profile — Eqs. 15–16.

    Parameters
    ----------
    pressure_bar : array   pressure grid [bar]
    log_delta    : float   log10(δ), where τ = δ · P_bar
    log_gamma    : float   log10(γ), visible-to-IR opacity ratio
    T_int        : float   internal temperature [K]
    T_eq         : float   equilibrium temperature [K]
    alpha        : float   upper-atmosphere modifier  (Eq. 15)
    log_P_trans  : float   log10(P_trans) [bar]
    """
    delta   = 10 ** log_delta
    gamma   = 10 ** log_gamma
    P_trans = 10 ** log_P_trans

    T_g   = _guillot(pressure_bar, delta, gamma, T_int, T_eq)
    T_mod = T_g * (1.0 - alpha / (1.0 + pressure_bar / P_trans))
    return _boxcar_logP(pressure_bar, T_mod)


print("petitRADTRANS functions loaded.")


In [ ]:
# ── pressure grid: 64 levels, 100 → 1e-7 bar (matches config) ────────────────
P = np.logspace(2, -7, 64)

# ── Line+2013 profiles — config bounds ───────────────────────────────────────
#   t_int_k          Normal(500, 20)
#   t_irr_k          Normal(1800, 500)
#   log10_kappa      Normal(-2.5, 2.5)      → kappa = 10^x  [m²/kg]
#   power_law_n      Uniform(0.5, 2.0)
#   log10_gamma_1/2  Uniform(-2, 2)         → gamma = 10^x
#   alpha            Uniform(0, 1)
#   t_shift_k        Uniform(-500, 500)
#   gravity_m_s2     Uniform(5, 50)         → 500–5000 cm/s²
#   convection       prob=0.333, adiabat ~ Uniform(0.25, 0.35)
# Rejection: resample if any level falls outside [0, 3000] K.

N   = 10
rng = np.random.default_rng()

def sample_line2013(rng):
    t_int   = rng.normal(500.0, 20.0)
    t_irr   = np.clip(rng.normal(1800.0, 500.0), 300.0, 4000.0)
    kappa   = 10 ** rng.normal(-2.5, 2.5)
    n       = rng.uniform(0.5, 2.0)
    gamma_1 = 10 ** rng.uniform(-2.0, 2.0)
    gamma_2 = 10 ** rng.uniform(-2.0, 2.0)
    alpha   = rng.uniform(0.0, 1.0)
    t_shift = rng.uniform(-500.0, 500.0)
    gravity = rng.uniform(5.0, 50.0)
    adiabat = rng.uniform(0.25, 0.35) if rng.random() < 0.333 else None
    return line2013_profile(P, t_int, t_irr, kappa, n, gamma_1, gamma_2,
                            alpha, gravity, adiabat, t_shift)

profiles_line = []
while len(profiles_line) < N:
    T = sample_line2013(rng)
    if T.min() >= 0.0 and T.max() <= 3000.0:
        profiles_line.append(T)

print(f"Generated {N} Line+2013 profiles")

In [ ]:
# ── petitRADTRANS profiles — config bounds ────────────────────────────────────
#   log_delta    derived from kappa_ir and gravity (config distributions)
#   log_gamma    Uniform(-2, 2)         log10_gamma_1_range
#   T_int        Normal(500, 20)        t_int_k_normal
#   T_eq         Normal(1800, 500)      t_irr_k_normal (used as T_eq)
#   alpha        Uniform(0, 1)          alpha_range
#   log_P_trans  Uniform(-5, 1)         (no direct config equivalent)
#   convection   prob=1/3, adiabat ~ Uniform(0.25, 0.35)
# Rejection: resample if any level falls outside [0, 3000] K.

def sample_molliere2019(rng):
    log_kappa   = rng.normal(-2.5, 2.5)
    log_g       = np.log10(rng.uniform(5.0, 50.0))
    log_delta   = log_kappa + 5.0 - log_g
    log_gamma   = rng.uniform(-2.0, 2.0)
    T_int       = rng.normal(500.0, 20.0)
    T_eq        = np.clip(rng.normal(1800.0, 500.0), 300.0, 4000.0)
    alpha       = rng.uniform(0.0, 1.0)
    log_P_trans = rng.uniform(-5.0, 1.0)
    T = molliere2019_pt_profile(P, log_delta, log_gamma, T_int, T_eq, alpha, log_P_trans)
    if rng.random() < 1/3:
        T = apply_convective_adjustment(P, T, rng.uniform(0.25, 0.35))
    return T

profiles_prt = []
while len(profiles_prt) < N:
    T = sample_molliere2019(rng)
    if T.min() >= 0.0 and T.max() <= 3000.0:
        profiles_prt.append(T)

print(f"Generated {N} petitRADTRANS Mollière+2019 profiles")


In [ ]:
colors = plt.cm.tab20(np.linspace(0, 1, N))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 7), sharey=True)
fig.subplots_adjust(wspace=0.05)

for i, T in enumerate(profiles_line):
    ax1.plot(T, P, lw=2.5, color=colors[i])
ax1.set_yscale("log")
ax1.invert_yaxis()
ax1.set_xlabel("Temperature (K)")
ax1.set_ylabel("Pressure (bar)")
ax1.set_title("Line+2013", fontsize=12)
ax1.grid(True, alpha=0.3)

for i, T in enumerate(profiles_prt):
    ax2.plot(T, P, lw=2.5, color=colors[i])
ax2.set_yscale("log")
ax2.set_xlabel("Temperature (K)")
ax2.set_title("Mollière+2019 (petitRADTRANS)", fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.tick_params(labelleft=False)

plt.tight_layout()
plt.show()
